In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Untuk mengaktifkan runtime

import pandas as pd
from pathlib import Path

NIM_KAMU = "2501010035"
base_path = Path(f"/content/drive/MyDrive/akomodasi_kintamani_2501010035")

csv_path = base_path / "data" / "interim" / "akomodasi_kintamani.csv"
akomodasi = pd.read_csv(csv_path)

print(f"Berhasil memuat data! Total baris: {len(akomodasi)}")

Berhasil memuat data! Total baris: 32


In [ ]:
from pathlib import Path

base_path = Path("/content/drive/MyDrive/akomodasi_kintamani_2501010035")

folders = [
    base_path / "data/raw",
    base_path / "data/interim",
    base_path / "notebooks",
    base_path / "docs",
    base_path / "reports"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Folder berhasil dibuat di Drive: {folder}")

Folder berhasil dibuat di Drive: /content/drive/MyDrive/akomodasi_kintamani_2501010035/data/raw
Folder berhasil dibuat di Drive: /content/drive/MyDrive/akomodasi_kintamani_2501010035/data/interim
Folder berhasil dibuat di Drive: /content/drive/MyDrive/akomodasi_kintamani_2501010035/notebooks
Folder berhasil dibuat di Drive: /content/drive/MyDrive/akomodasi_kintamani_2501010035/docs
Folder berhasil dibuat di Drive: /content/drive/MyDrive/akomodasi_kintamani_2501010035/reports


In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests

NIM_KAMU = "2501010035"
base_path = Path(f"/content/drive/MyDrive/akomodasi_kintamani_2501010035")

URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": f"INSTIKI-BIFP243-Praktikum/1.0 (2501010035@instiki.ac.id)"}

QUERY = """
[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;
"""

def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            print(f"[percobaan {percobaan}] status {r.status_code}, tunggu {jeda}s")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")
        time.sleep(jeda)
        jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass")

print("Sedang mengambil data akomodasi Kintamani...")
data = ambil_overpass(QUERY)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
raw_dir = base_path / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / f"osm_akomodasi_kintamani_{stamp}.json"
raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Sukses! Total data didapat: {len(data['elements'])} elemen.")
print(f"File mentah tersimpan aman di: {raw_path}")

# Jalankan query cukup satu atau dua kali.

Sedang mengambil data akomodasi Kintamani...
[percobaan 1] status 502, tunggu 5s
[percobaan 2] status 502, tunggu 10s
Sukses! Total data didapat: 32 elemen.
File mentah tersimpan aman di: /content/drive/MyDrive/akomodasi_kintamani_2501010035/data/raw/osm_akomodasi_kintamani_20261108_145017.json


In [12]:
import pandas as pd
import json
from pathlib import Path
import glob

def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)

        baris.append({
            "osm_type": e["type"],
            "osm_id": e["id"],
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris)

NIM_KAMU = "2501010035"
base_path = Path(f"/content/drive/MyDrive/akomodasi_kintamani_2501010035")

if 'data' not in globals():
    print("Variabel 'data' tidak ditemukan di memori. Mencoba memuat dari file raw JSON di Drive...")
    raw_dir = base_path / "data" / "raw"
    json_files = sorted(raw_dir.glob("osm_akomodasi_kintamani_*.json"))
    if json_files:
        terbaru = json_files[-1]
        print(f"Memuat data dari: {terbaru.name}")
        with open(terbaru, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        raise FileNotFoundError("File JSON mentah tidak ditemukan di Drive. Silakan jalankan sel pengunduh data terlebih dahulu.")

akomodasi = ratakan(data["elements"])

print(akomodasi["jenis"].value_counts())
print("Tanpa nama:", akomodasi['nama'].isna().sum())

interim_dir = base_path / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
csv_path = interim_dir / "akomodasi_kintamani.csv"
akomodasi.to_csv(csv_path, index=False, encoding="utf-8")

print(f"\nFile CSV berhasil disimpan di: {csv_path}")
display(akomodasi.head())

jenis
hotel          16
guest_house     9
hostel          4
chalet          2
motel           1
Name: count, dtype: int64
Tanpa nama: 1

File CSV berhasil disimpan di: /content/drive/MyDrive/akomodasi_kintamani_2501010035/data/interim/akomodasi_kintamani.csv


,osm_type,osm_id,nama,jenis,lat,lon,desa,bintang,jumlah_kamar,website,url_osm
0,node,1977779907,The Lakeview Hotel Bali,hotel,-8.283140,115.365844,None,None,None,http://lakeviewbali.com/en/,https://www.openstreetmap.org/node/1977779907
1,node,2107226701,Arlina,guest_house,-8.249815,115.398465,None,None,None,None,https://www.openstreetmap.org/node/2107226701
2,node,2107227001,Toya Devasya Resort,hotel,-8.250164,115.399798,None,None,None,None,https://www.openstreetmap.org/node/2107227001
3,node,4150326329,Guesthouse,hotel,-8.281456,115.362223,None,None,None,None,https://www.openstreetmap.org/node/4150326329
4,node,4297310696,The Cave Hotel,hotel,-8.280748,115.362048,None,2,5,None,https://www.openstreetmap.org/node/4297310696


In [ ]:
import pandas as pd

print("=== 1. PEMERIKSAAN DUPLIKAT ===" )
duplikat = akomodasi.duplicated(subset=['osm_type', 'osm_id']).sum()
print(f"Jumlah baris duplikat (berdasarkan osm_type + osm_id): {duplikat}")

print("\n=== 2. PEMERIKSAAN DATA TANPA NAMA ===")
tanpa_nama = akomodasi['nama'].isna().sum()
print(f"Jumlah akomodasi yang tidak memiliki nama (NaN): {tanpa_nama}")

print("\n=== 3. PEMERIKSAAN KOLOM KOSONG (NULL VALUES) ===")
print(akomodasi.isnull().sum())

print("\n=== 4. PEMERIKSAAN KOORDINAT DI LUAR WILAYAH KINTAMANI ===")
# Batasan kasar wilayah Kintamani, Bangli, Bali (Latitude: -8.5 s.d -8.0, Longitude: 115.2 s.d 115.6)
# Kita cek apakah ada koordinat yang keluar dari rentang tersebut atau bernilai null
lat_invalid = akomodasi[(akomodasi['lat'] < -8.5) | (akomodasi['lat'] > -8.0) | (akomodasi['lat'].isna())]
lon_invalid = akomodasi[(akomodasi['lon'] < 115.2) | (akomodasi['lon'] > 115.6) | (akomodasi['lon'].isna())]
print(f"Jumlah baris dengan Latitude di luar batas/kosong: {len(lat_invalid)}")
print(f"Jumlah baris dengan Longitude di luar batas/kosong: {len(lon_invalid)}")

=== 1. PEMERIKSAAN DUPLIKAT ===
Jumlah baris duplikat (berdasarkan osm_type + osm_id): 0

=== 2. PEMERIKSAAN DATA TANPA NAMA ===
Jumlah akomodasi yang tidak memiliki nama (NaN): 1

=== 3. PEMERIKSAAN KOLOM KOSONG (NULL VALUES) ===
osm_type         0
osm_id           0
nama             1
jenis            0
lat              0
lon              0
desa            32
bintang         31
jumlah_kamar    30
website         30
url_osm          0
dtype: int64

=== 4. PEMERIKSAAN KOORDINAT DI LUAR WILAYAH KINTAMANI ===
Jumlah baris dengan Latitude di luar batas/kosong: 0
Jumlah baris dengan Longitude di luar batas/kosong: 0


In [6]:
import pandas as pd

jumlah_osm = len(akomodasi)
print(f"Jumlah akomodasi dari OSM: {jumlah_osm}")

jumlah_bps = 78
tahun_bps = "2025"
sumber_bps = "BPS Kabupaten Bangli - Kecamatan Kintamani Dalam Angka 2025"

persentase = (jumlah_osm / jumlah_bps) * 100
print(f"Jumlah data BPS ({tahun_bps}): {jumlah_bps}")
print(f"Sumber Acuan: {sumber_bps}")
print(f"Persentase Kelengkapan: {persentase:.2f}%")

print("\n=== 10 SAMPEL ACAK UNTUK VERIFIKASI MANUAL ===")
sampel_10 = akomodasi.sample(n=min(10, len(akomodasi)), random_state=42)
for idx, row in sampel_10.iterrows():
    print(f"- Nama: {row['nama']} | Jenis: {row['jenis']} | Link: {row['url_osm']}")

Jumlah akomodasi dari OSM: 32
Jumlah data BPS (2025): 78
Sumber Acuan: BPS Kabupaten Bangli - Kecamatan Kintamani Dalam Angka 2025
Persentase Kelengkapan: 41.03%

=== 10 SAMPEL ACAK UNTUK VERIFIKASI MANUAL ===
- Nama: Batur Sunrise Guesthouse | Jenis: hotel | Link: https://www.openstreetmap.org/node/13068182153
- Nama: Caldera | Jenis: hotel | Link: https://www.openstreetmap.org/node/6451134585
- Nama: Bunbulan Panorama Villas | Jenis: guest_house | Link: https://www.openstreetmap.org/node/9769685089
- Nama: Segara Hotel | Jenis: hotel | Link: https://www.openstreetmap.org/node/7058054537
- Nama: The Volcania Guesthouse | Jenis: hotel | Link: https://www.openstreetmap.org/node/5065920543
- Nama: Entrance to: Bunbulan Hill Hostel | Jenis: hostel | Link: https://www.openstreetmap.org/node/5175562421
- Nama: Lava Bali and Hot Spring | Jenis: hotel | Link: https://www.openstreetmap.org/node/13284479326
- Nama: Villa Jempana | Jenis: hotel | Link: https://www.openstreetmap.org/node/97787359

In [7]:

bps_update_df = pd.DataFrame([
    {"sumber": "OpenStreetMap (OSM)", "jumlah": len(akomodasi), "tahun": "2026"},
    {"sumber": "Badan Pusat Statistik (BPS) Kintamani Dalam Angka", "jumlah": jumlah_bps, "tahun": "2025"}
])
bps_update_df.to_csv(base_path / "docs" / "perbandingan_bps.csv", index=False, encoding="utf-8")
print("File perbandingan_bps.csv di folder docs/ berhasil diperbarui!")

File perbandingan_bps.csv di folder docs/ berhasil diperbarui!


In [5]:
from pathlib import Path
import json
import pandas as pd

NIM_KAMU = "2501010035"
base_path = Path(f"/content/drive/MyDrive/akomodasi_kintamani_2501010035")
docs_dir = base_path / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

query_text = """[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;"""
(docs_dir / "query_overpass.txt").write_text(query_text, encoding="utf-8")

provenance_data = {
    "source": "OpenStreetMap",
    "api": "Overpass API",
    "region": "Kintamani, Bangli, Bali",
    "nim": NIM_KAMU
}
(docs_dir / "provenance.json").write_text(json.dumps(provenance_data, indent=2, ensure_ascii=False), encoding="utf-8")

dict_df = pd.DataFrame([
    {"kolom": "osm_type", "tipe_data": "string", "deskripsi": "Tipe elemen OSM (node/way)"},
    {"kolom": "osm_id", "tipe_data": "int", "deskripsi": "ID unik elemen pada OSM"},
    {"kolom": "nama", "tipe_data": "string", "deskripsi": "Nama akomodasi"},
    {"kolom": "jenis", "tipe_data": "string", "deskripsi": "Kategori pariwisata (hotel, guest_house, dll)"},
    {"kolom": "lat", "tipe_data": "float", "deskripsi": "Titik lintang (latitude)"},
    {"kolom": "lon", "tipe_data": "float", "deskripsi": "Titik bujur (longitude)"},
    {"kolom": "desa", "tipe_data": "string", "deskripsi": "Nama desa/kelurahan"},
    {"kolom": "bintang", "tipe_data": "string", "deskripsi": "Klasifikasi bintang akomodasi"},
    {"kolom": "jumlah_kamar", "tipe_data": "string", "deskripsi": "Jumlah kamar"},
    {"kolom": "website", "tipe_data": "string", "deskripsi": "Situs web resmi akomodasi"},
    {"kolom": "url_osm", "tipe_data": "string", "deskripsi": "Tautan langsung ke elemen OSM"}
])
dict_df.to_csv(docs_dir / "data_dictionary.csv", index=False, encoding="utf-8")

ethics_df = pd.DataFrame([
    {"prinsip": "Minimalisasi Data", "status": "Ya", "keterangan": "Nomor telepon dan email sengaja tidak diambil demi privasi"},
    {"prinsip": "Penggunaan API yang Etis", "status": "Ya", "keterangan": "Menggunakan jeda dan tidak melakukan spam request ke server"},
    {"prinsip": "Integritas Data", "status": "Ya", "keterangan": "Tidak ada data yang dihapus meskipun terdapat nilai kosong"}
])
ethics_df.to_csv(docs_dir / "data_ethics_checklist.csv", index=False, encoding="utf-8")

bps_update_df = pd.DataFrame([
    {"sumber": "OpenStreetMap (OSM)", "jumlah": len(akomodasi), "tahun": "2026"},
    {"sumber": "Badan Pusat Statistik (BPS) Kintamani Dalam Angka", "jumlah": 78, "tahun": "2025"}
])
bps_update_df.to_csv(docs_dir / "perbandingan_bps.csv", index=False, encoding="utf-8")

print("Semua file dokumentasi di dalam folder docs/ berhasil dibuat lengkap!")

Semua file dokumentasi di dalam folder docs/ berhasil dibuat lengkap!


In [5]:
from pathlib import Path

NIM_KAMU = "2501010035"
base_path = Path(f"/content/drive/MyDrive/akomodasi_kintamani_2501010035")

base_path.mkdir(parents=True, exist_ok=True)

readme_content = """# Akuisisi Data Akomodasi Kecamatan Kintamani

Repositori ini dikembangkan untuk menyelesaikan tugas praktikum mata kuliah **BIFP-243 Akuisisi dan Manajemen Data** dengan studi kasus ekstraksi, pembersihan, dan analisis data spasial akomodasi pariwisata di Kecamatan Kintamani, Kabupaten Bangli, Bali.

## Identitas Mahasiswa
* **Nama:** Hanifah Aulia Ramadhani
* **NIM:** 2501010035
* **Mata Kuliah:** BIFP-243 Akuisisi dan Manajemen Data
* **Institusi:** Institut Bisnis dan Teknologi Indonesia (INSTIKI)

## Tujuan
Memahami proses pengambilan data geospasial menggunakan API, mengolah data mentah bertingkat menjadi format tabular DataFrame, memeriksa kualitas dan kelengkapan data (*completeness rate*), serta menyusun laporan analisis spasial persebaran akomodasi di Kecamatan Kintamani.

## Sumber Data & Tanggal Akses
1. **OpenStreetMap (OSM)** melalui **Overpass API** (Data spasial primer titik dan area pariwisata).
2. **Badan Pusat Statistik (BPS) Kabupaten Bangli** melalui publikasi *Kecamatan Kintamani Dalam Angka 2025* (Data pembanding statistik resmi).
* **Tanggal Akses Data:** November 2026.

## Metode Akuisisi
Proses ekstraksi dan pemrosesan data dilakukan melalui tahapan berikut:
1. Mengambil data spasial akomodasi menggunakan kueri Overpass API.
2. Menerapkan mekanisme tangguh (*retry* / penanganan pengecualian) apabila permintaan API mengalami kendala jaringan.
3. Menyimpan data mentah hasil respons API dalam format JSON ke direktori `data/raw/`.
4. Menerapkan fungsi transformasi kustom (`ratakan()`) untuk mengubah struktur JSON bertingkat (`node` dan `way`) menjadi tabel tabular menggunakan Pandas.
5. Memeriksa kualitas data, nilai kosong (*missing values*), serta melakukan uji kelengkapan (*completeness analysis*) terhadap data acuan BPS.
6. Melakukan verifikasi acak terhadap sampel data (*ground truthing audit*).
7. Membuat visualisasi grafik persebaran akomodasi di Kecamatan Kintamani.

## Batasan dan Etika Data
- Menggunakan sumber data terbuka (*open data*).
- Tidak melakukan *scraping* ilegal dari platform pihak ketiga tertutup seperti Online Travel Agent (OTA) maupun Google Maps.
- Menerapkan prinsip **Minimalisasi Data**: tidak mengambil, menyimpan, atau memproses data kontak pribadi seperti nomor telepon atau alamat email pengelola.
- Mematahi batasan teknis penggunaan API (*rate limiting*) untuk menghindari beban berlebih pada peladen OSM.
- Tetap mencantumkan atribusi kepemilikan data secara transparan.

*Atribusi: © OpenStreetMap contributors. Lisensi Open Database License (ODbL).*

## Cara Menjalankan Ulang
1. Instal pustaka (*libraries*) yang diperlukan:
   ```bash
   pip install requests pandas matplotlib jupyter
   ```
"""

# Menyimpan konten ke file README.md
readme_path = base_path / "README.md"
readme_path.write_text(readme_content, encoding="utf-8")
print(f"File README.md berhasil dibuat di: {readme_path}")

File README.md berhasil dibuat di: /content/drive/MyDrive/akomodasi_kintamani_2501010035/README.md
